# 03 — Unification des sources

**Projet** : FakeNews Analyzer — DevComplex  
**Objectif** : Fusionner toutes les sources nettoyées en un schéma commun, split train/test stratifié.

**Exécuter après** : `02_cleaning.ipynb`

---

In [1]:
# ============================================================
# Fichier  : 03_unification.ipynb
# Rôle     : Fusion de toutes les sources en schéma commun
# Version  : V1 (local)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================

import sys, os
sys.path.insert(0, '..')

import glob
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

from spark_utils import (
    get_spark_session, show_label_distribution,
    save_parquet, stratified_split, check_class_balance
)

spark = get_spark_session(app_name='FakeNews-Unification', memory='8g')
PROCESSED_DIR = '../09_data/processed'

print('Session Spark démarrée')

Session Spark démarrée


## Section 1 — Chargement des fichiers nettoyés

In [2]:
parquet_files = glob.glob(os.path.join(PROCESSED_DIR, 'cleaned_*.parquet'))

if not parquet_files:
    raise FileNotFoundError(
        f'Aucun fichier cleaned_*.parquet trouvé dans {PROCESSED_DIR}.\n'
        "Exécuter 02_cleaning.ipynb d'abord."
    )

print(f'Parquet trouvés : {len(parquet_files)}')
for p in parquet_files:
    print(f'  {os.path.basename(p)}')

dataframes = {}
for p in parquet_files:
    source_name = os.path.basename(p).replace('cleaned_', '').replace('.parquet', '')
    df = spark.read.parquet(p)
    dataframes[source_name] = df
    print(f'  Chargé : {source_name} — {df.count():,} lignes | colonnes : {df.columns}')

Parquet trouvés : 4
  cleaned_fake_news.parquet
  cleaned_isot_fake.parquet
  cleaned_isot_true.parquet
  cleaned_welfake.parquet
  Chargé : fake_news — 38,234 lignes | colonnes : ['clean_text', 'label', 'source']
  Chargé : isot_fake — 17,313 lignes | colonnes : ['clean_text', 'label', 'source']
  Chargé : isot_true — 20,921 lignes | colonnes : ['clean_text', 'label', 'source']
  Chargé : welfake — 61,987 lignes | colonnes : ['clean_text', 'label', 'source']


## Section 2 — Harmonisation du schéma commun

In [3]:
unified_dfs = []

for name, df in dataframes.items():
    print(f'\n── Harmonisation : {name}')
    
    if 'clean_text' not in df.columns:
        text_col = next((c for c in df.columns if 'text' in c.lower()), None)
        if text_col:
            df = df.withColumnRenamed(text_col, 'clean_text')
        else:
            print(f'  ⚠ Impossible de trouver une colonne texte — ignoré')
            continue
    
    if 'label' in df.columns:
        df = df.withColumn('label', F.col('label').cast(IntegerType()))
        df = df.filter(F.col('label').isin(0, 1))
    else:
        print(f'  ⚠ Pas de colonne label — toutes les lignes étiquetées REAL (0)')
        df = df.withColumn('label', F.lit(0))
    
    if 'source' not in df.columns:
        df = df.withColumn('source', F.lit(name))
    
    df = df.withColumn('text_len', F.size(F.split(F.col('clean_text'), ' ')))
    df = df.select('clean_text', 'label', 'source', 'text_len')
    
    count = df.count()
    print(f'  {count:,} lignes harmonisées')
    unified_dfs.append(df)


── Harmonisation : fake_news
  38,234 lignes harmonisées

── Harmonisation : isot_fake
  17,313 lignes harmonisées

── Harmonisation : isot_true
  20,921 lignes harmonisées

── Harmonisation : welfake
  61,987 lignes harmonisées


## Section 3 — Union et déduplication

In [4]:
if not unified_dfs:
    raise ValueError('Aucun DataFrame harmonisé disponible. Vérifiez les étapes précédentes.')

unified = unified_dfs[0]
for df in unified_dfs[1:]:
    unified = unified.union(df)

total = unified.count()
print(f'\nDataFrame unifié : {total:,} lignes')
print(f'Colonnes : {unified.columns}')


DataFrame unifié : 138,455 lignes
Colonnes : ['clean_text', 'label', 'source', 'text_len']


## Section 4 — Déduplication finale

In [5]:
before = unified.count()
unified = unified.dropDuplicates(['clean_text'])
after = unified.count()
print(f'Déduplication finale : {before - after:,} doublons supprimés → {after:,} lignes')

Déduplication finale : 76,468 doublons supprimés → 61,987 lignes


## Section 5 — Rapport de fusion

In [6]:
total = unified.count()

source_stats = (
    unified
    .groupBy('source', 'label')
    .count()
    .toPandas()
    .pivot(index='source', columns='label', values='count')
    .fillna(0)
    .astype(int)
)

if 0 in source_stats.columns and 1 in source_stats.columns:
    source_stats.columns = ['REAL', 'FAKE']
    source_stats['total'] = source_stats['REAL'] + source_stats['FAKE']
    source_stats['FAKE%'] = (source_stats['FAKE'] / source_stats['total'] * 100).round(1)
    source_stats['REAL%'] = (source_stats['REAL'] / source_stats['total'] * 100).round(1)

print('\n=== RAPPORT DE FUSION ===')
print(source_stats.to_string())
print(f'\nTOTAL : {total:,} articles')


=== RAPPORT DE FUSION ===
            REAL   FAKE  total  FAKE%  REAL%
source                                      
fake_news  20921  17313  38234   45.3   54.7
welfake    13425  10328  23753   43.5   56.5

TOTAL : 61,987 articles


## Section 6 — Vérification de l'équilibre et split train/test

In [7]:
check_class_balance(unified, label_col='label', min_minority_ratio=0.35)
show_label_distribution(unified, label_col='label')


Distribution des labels (total : 61,987)
----------------------------------------
+-----+-----+-----------+
|label|count|pourcentage|
+-----+-----+-----------+
|0    |34346|55.41      |
|1    |27641|44.59      |
+-----+-----+-----------+



In [8]:
train_df, test_df = stratified_split(unified, label_col='label', test_size=0.2, seed=42)

print(f'\nSplit train/test :')
print(f'  Train : {train_df.count():,}')
print(f'  Test  : {test_df.count():,}')

print('\nDistribution TRAIN :')
show_label_distribution(train_df)
print('\nDistribution TEST :')
show_label_distribution(test_df)


Split train/test :
  Train : 49,919
  Test  : 12,068

Distribution TRAIN :

Distribution des labels (total : 49,919)
----------------------------------------
+-----+-----+-----------+
|label|count|pourcentage|
+-----+-----+-----------+
|0    |27645|55.38      |
|1    |22274|44.62      |
+-----+-----+-----------+


Distribution TEST :

Distribution des labels (total : 12,068)
----------------------------------------
+-----+-----+-----------+
|label|count|pourcentage|
+-----+-----+-----------+
|0    |6701 |55.53      |
|1    |5367 |44.47      |
+-----+-----+-----------+



## Section 7 — Sauvegarde

In [9]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

save_parquet(unified,  os.path.join(PROCESSED_DIR, 'unified.parquet'))
save_parquet(train_df, os.path.join(PROCESSED_DIR, 'train.parquet'))
save_parquet(test_df,  os.path.join(PROCESSED_DIR, 'test.parquet'))

print('\n✓ Fichiers sauvegardés :')
print(f'  {PROCESSED_DIR}/unified.parquet')
print(f'  {PROCESSED_DIR}/train.parquet')
print(f'  {PROCESSED_DIR}/test.parquet')
print('\nProchaine étape : exécuter 04_feature_engineering.ipynb')

spark.stop()

✓ Sauvegardé : ../09_data/processed\unified.parquet
✓ Sauvegardé : ../09_data/processed\train.parquet
✓ Sauvegardé : ../09_data/processed\test.parquet

✓ Fichiers sauvegardés :
  ../09_data/processed/unified.parquet
  ../09_data/processed/train.parquet
  ../09_data/processed/test.parquet

Prochaine étape : exécuter 04_feature_engineering.ipynb
